In [2]:
# CELL 1: NeuroLens — Shared Google Drive Setup

from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive
drive.mount("/content/drive")

# IMPORTANT:
# Adjusting path to handle the nested shortcut folder structure
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Mega Project/Mega Project"
)

# Shared folders
GLOBAL_DIR = PROJECT_DIR / "global_models"
CLIENT_DIR = PROJECT_DIR / "hospital_outputs"
LOG_DIR = PROJECT_DIR / "training_logs"

# Create directories if they do not exist
for folder in [PROJECT_DIR, GLOBAL_DIR, CLIENT_DIR, LOG_DIR]:
    try:
        folder.mkdir(parents=True, exist_ok=True)
    except OSError as e:
        print(f"Error creating directory {folder}: {e}")
        print("Please verify that Google Drive is mounted correctly and the path is accessible.")

print("=" * 60)
print("NEUROLENS — SHARED TRAINING ENVIRONMENT")
print("=" * 60)

print("Project directory :", PROJECT_DIR)
print("Global models      :", GLOBAL_DIR)
print("Hospital outputs   :", CLIENT_DIR)
print("Training logs      :", LOG_DIR)

try:
    print("\nProject folder exists:", PROJECT_DIR.exists())
    print("Global folder exists:", GLOBAL_DIR.exists())
except Exception as e:
    print("\nCould not verify folder existence:", e)

print("\nSetup complete.")

Mounted at /content/drive
NEUROLENS — SHARED TRAINING ENVIRONMENT
Project directory : /content/drive/MyDrive/Mega Project/Mega Project
Global models      : /content/drive/MyDrive/Mega Project/Mega Project/global_models
Hospital outputs   : /content/drive/MyDrive/Mega Project/Mega Project/hospital_outputs
Training logs      : /content/drive/MyDrive/Mega Project/Mega Project/training_logs

Project folder exists: True
Global folder exists: True

Setup complete.


In [3]:
# CELL 3: Verify Shared Global Model
# ALL THREE HOSPITALS RUN THIS CELL

import torch
import json
import hashlib

GLOBAL_CHECKPOINT = GLOBAL_DIR / "global_model_round_000.pth"
GLOBAL_METADATA = GLOBAL_DIR / "global_model_round_000.json"

assert GLOBAL_CHECKPOINT.exists(), (
    "Global checkpoint not found. Check shared Drive access."
)
assert GLOBAL_METADATA.exists(), (
    "Global metadata not found. Run Cell 2 in Hospital 1 first."
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

with open(GLOBAL_METADATA, "r", encoding="utf-8") as f:
    metadata = json.load(f)

actual_hash = sha256_file(GLOBAL_CHECKPOINT)

assert actual_hash == metadata["checkpoint_sha256"], (
    "HASH MISMATCH! Do not train until this is resolved."
)

global_state = torch.load(
    GLOBAL_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

assert "features.conv0.weight" in global_state
assert "classifier.weight" in global_state
assert "classifier.bias" in global_state
assert global_state["classifier.weight"].shape == (4, 1024)

print("=" * 60)
print("GLOBAL MODEL VERIFICATION PASSED")
print("=" * 60)
print("Model:", metadata["model_name"])
print("Round:", metadata["initial_round"])
print("Checkpoint:", GLOBAL_CHECKPOINT.name)
print("Classifier shape:", tuple(global_state["classifier.weight"].shape))
print("SHA-256:", actual_hash)

GLOBAL MODEL VERIFICATION PASSED
Model: DenseNet121
Round: 0
Checkpoint: global_model_round_000.pth
Classifier shape: (4, 1024)
SHA-256: 5c35dbf9eafa1ef1c923b603d2d44318aadd3abf2756275b0fd159ff0c9cc754


In [4]:
# CELL 4: Locate and Inspect the Shared Dataset ZIP
# ALL THREE MEMBERS RUN THIS CELL

from pathlib import Path
import zipfile

PROJECT_DIR = Path("/content/drive/MyDrive/Mega Project/Mega Project")

ZIP_PATH = PROJECT_DIR / "federated_split.zip"

assert ZIP_PATH.exists(), (
    f"ZIP file not found: {ZIP_PATH}\n"
    "Check the filename in your shared Google Drive."
)

print("=" * 65)
print("NEUROLENS — DATASET ZIP INSPECTION")
print("=" * 65)
print("ZIP file:", ZIP_PATH)
print("Size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 2))

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = [name for name in z.namelist() if not name.endswith("/")]

    print("\nTotal files inside ZIP:", len(files))
    print("\nFirst 40 file paths:")

    for name in files[:40]:
        print(name)

    # Identify top-level folders
    top_level = sorted({
        name.split("/")[0]
        for name in files
        if "/" in name
    })

    print("\nTop-level folders:")
    for folder in top_level:
        print(" -", folder)

print("\nInspection complete. No files extracted yet.")

NEUROLENS — DATASET ZIP INSPECTION
ZIP file: /content/drive/MyDrive/Mega Project/Mega Project/federated_split.zip
Size (MB): 1283.05

Total files inside ZIP: 87414

First 40 file paths:
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_100.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_101.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_102.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_103.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_104.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_105.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_106.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_107.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_108.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_109.jpg
federated_split/hospital_1/test/Mild Dementia/OAS1_0269_MR1_mpr-1_110.jpg


In [5]:
# CELL 5: Extract the Assigned Hospital Dataset
# ALL THREE MEMBERS RUN THIS CELL
# IMPORTANT: Change HOSPITAL_ID for your own notebook.

from pathlib import Path
import zipfile

PROJECT_DIR = Path("/content/drive/MyDrive/Mega Project/Mega Project")
ZIP_PATH = PROJECT_DIR / "federated_split.zip"

# CHANGE THIS LINE IN EACH NOTEBOOK:
HOSPITAL_ID = "hospital_2"  # hospital_1 / hospital_2 / hospital_3

EXTRACT_ROOT = Path("/content/neurolens_data")
DATASET_DIR = EXTRACT_ROOT / "federated_split" / HOSPITAL_ID

assert HOSPITAL_ID in {
    "hospital_1", "hospital_2", "hospital_3"
}, "Invalid hospital ID."

assert ZIP_PATH.exists(), f"ZIP not found: {ZIP_PATH}"

# Check whether extraction has already completed.
required_splits = ["train", "val", "test"]
if all((DATASET_DIR / split).is_dir() for split in required_splits):
    print("Hospital dataset folders already exist.")
    print("Skipping extraction:", DATASET_DIR)

else:
    prefix = f"federated_split/{HOSPITAL_ID}/"

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        members = [
            info for info in z.infolist()
            if not info.is_dir()
            and info.filename.startswith(prefix)
        ]

        assert members, (
            f"No files found for {HOSPITAL_ID} inside the ZIP."
        )

        print(f"Extracting {HOSPITAL_ID}")
        print("Files to extract:", len(members))

        # ZIP paths are controlled by the archive's known prefix.
        for info in members:
            relative_path = info.filename[len(prefix):]
            destination = DATASET_DIR / relative_path

            # Guard against unsafe paths.
            if not relative_path or Path(relative_path).is_absolute():
                raise ValueError(f"Unsafe archive path: {info.filename}")

            if ".." in Path(relative_path).parts:
                raise ValueError(f"Unsafe archive path: {info.filename}")

            destination.parent.mkdir(parents=True, exist_ok=True)

            with z.open(info) as source, open(destination, "wb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

    print("Extraction complete.")

print("\nHospital:", HOSPITAL_ID)
print("Dataset path:", DATASET_DIR)

for split in required_splits:
    split_path = DATASET_DIR / split
    print(f"{split:12s}: exists = {split_path.is_dir()}")

assert all((DATASET_DIR / s).is_dir() for s in required_splits), (
    "Expected train, validation, and test folders were not all found. "
    "Stop and send me the output."
)

print("\nCELL 5 PASSED — hospital dataset extracted.")

Extracting hospital_2
Files to extract: 29362
Extraction complete.

Hospital: hospital_2
Dataset path: /content/neurolens_data/federated_split/hospital_2
train       : exists = True
val         : exists = True
test        : exists = True

CELL 5 PASSED — hospital dataset extracted.


In [6]:
# CELL 6: Verify Hospital Dataset Structure
# ALL THREE MEMBERS RUN THIS CELL

from pathlib import Path
from collections import Counter

# Change this for your notebook
HOSPITAL_ID = "hospital_2"  # hospital_1 / hospital_2 / hospital_3

DATASET_DIR = (
    Path("/content/neurolens_data/federated_split")
    / HOSPITAL_ID
)

SPLITS = ["train", "val", "test"]

EXPECTED_CLASSES = {
    "Mild Dementia",
    "Moderate Dementia",
    "Non Demented",
    "Very mild Dementia",
}

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".webp", ".tif", ".tiff"
}

print("=" * 65)
print(f"NEUROLENS — DATASET VALIDATION: {HOSPITAL_ID}")
print("=" * 65)

assert DATASET_DIR.exists(), f"Dataset not found: {DATASET_DIR}"

counts = {}

for split in SPLITS:
    split_dir = DATASET_DIR / split

    assert split_dir.is_dir(), (
        f"Missing '{split}' folder at {split_dir}"
    )

    class_folders = {
        p.name: p for p in split_dir.iterdir()
        if p.is_dir()
    }

    found_classes = set(class_folders.keys())

    print(f"\n[{split.upper()}]")

    if found_classes != EXPECTED_CLASSES:
        print("Expected classes:", sorted(EXPECTED_CLASSES))
        print("Found classes   :", sorted(found_classes))
        raise ValueError(
            f"Class-folder mismatch in {split}. "
            "Stop and send me the folder names."
        )

    counts[split] = {}

    for class_name in sorted(EXPECTED_CLASSES):
        count = sum(
            1 for p in class_folders[class_name].rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        )

        counts[split][class_name] = count
        print(f"{class_name:25s}: {count:,}")

        assert count > 0, (
            f"No images found for '{class_name}' in {split}."
        )

    print(f"{'TOTAL':25s}: {sum(counts[split].values()):,}")

print("\n" + "=" * 65)
print("DATASET STRUCTURE VALIDATION PASSED")
print("=" * 65)
print("Hospital:", HOSPITAL_ID)
print("Splits:", ", ".join(SPLITS))
print("Class count:", len(EXPECTED_CLASSES))
print("No model training has started.")

NEUROLENS — DATASET VALIDATION: hospital_2

[TRAIN]
Mild Dementia            : 1,220
Moderate Dementia        : 260
Non Demented             : 18,605
Very mild Dementia       : 3,538
TOTAL                    : 23,623

[VAL]
Mild Dementia            : 244
Moderate Dementia        : 33
Non Demented             : 2,196
Very mild Dementia       : 488
TOTAL                    : 2,961

[TEST]
Mild Dementia            : 244
Moderate Dementia        : 33
Non Demented             : 2,074
Very mild Dementia       : 427
TOTAL                    : 2,778

DATASET STRUCTURE VALIDATION PASSED
Hospital: hospital_2
Splits: train, val, test
Class count: 4
No model training has started.


In [7]:
# CELL 7: Image Preprocessing and DataLoaders
# ALL THREE MEMBERS RUN THIS CELL

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path
import os

# Change for your own notebook
HOSPITAL_ID = "hospital_2"  # hospital_1 / hospital_2 / hospital_3

DATASET_DIR = (
    Path("/content/neurolens_data/federated_split")
    / HOSPITAL_ID
)

# Reproducibility
SEED = 42
torch.manual_seed(SEED)

# Use GPU if available
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Shared model's expected class order.
# IMPORTANT: This must remain identical at every hospital.
CLASS_NAMES = [
    "Mild Dementia",
    "Moderate Dementia",
    "Non Demented",
    "Very mild Dementia",
]

NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# ImageNet normalization for pretrained DenseNet121
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# No random augmentation yet: use consistent preprocessing
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load datasets from train / val / test folders
train_dataset = datasets.ImageFolder(
    DATASET_DIR / "train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    DATASET_DIR / "val",
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    DATASET_DIR / "test",
    transform=eval_transform
)

# Verify class mappings are identical across splits
expected_mapping = {
    name: index for index, name in enumerate(CLASS_NAMES)
}

for split_name, dataset in [
    ("train", train_dataset),
    ("val", val_dataset),
    ("test", test_dataset),
]:
    assert dataset.class_to_idx == expected_mapping, (
        f"{split_name} class mapping differs!\n"
        f"Expected: {expected_mapping}\n"
        f"Found: {dataset.class_to_idx}"
    )

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("=" * 65)
print(f"NEUROLENS — DATALOADERS READY: {HOSPITAL_ID}")
print("=" * 65)
print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Class mapping:", train_dataset.class_to_idx)
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

# Test one batch to catch loading or preprocessing errors
images, labels = next(iter(train_loader))

print("\nSample batch shape:", tuple(images.shape))
print("Sample labels shape:", tuple(labels.shape))
print("Sample image dtype:", images.dtype)
print("Data loading successful.")

NEUROLENS — DATALOADERS READY: hospital_2
Device: cuda
Image size: 224
Batch size: 32
Class mapping: {'Mild Dementia': 0, 'Moderate Dementia': 1, 'Non Demented': 2, 'Very mild Dementia': 3}
Train images: 23623
Validation images: 2961
Test images: 2778

Sample batch shape: (32, 3, 224, 224)
Sample labels shape: (32,)
Sample image dtype: torch.float32
Data loading successful.


In [9]:
# CELL 8: Load and Verify the Shared Global Model
# ALL THREE MEMBERS RUN THIS CELL

import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path
import hashlib
import json

# Change this for your notebook
HOSPITAL_ID = "hospital_2"  # hospital_1 / hospital_2 / hospital_3

GLOBAL_DIR = Path("/content/drive/MyDrive/Mega Project/Mega Project/global_models")
GLOBAL_CHECKPOINT = GLOBAL_DIR / "global_model_round_000.pth"
GLOBAL_METADATA = GLOBAL_DIR / "global_model_round_000.json"

# 1. Verify the checkpoint file and metadata exist
assert GLOBAL_CHECKPOINT.exists(), "Round 0 checkpoint is missing."
assert GLOBAL_METADATA.exists(), "Round 0 metadata is missing."

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

with open(GLOBAL_METADATA, "r", encoding="utf-8") as f:
    metadata = json.load(f)

checkpoint_hash = sha256_file(GLOBAL_CHECKPOINT)

assert checkpoint_hash == metadata["checkpoint_sha256"], (
    "Checkpoint hash mismatch. STOP."
)

# 2. Create the exact same DenseNet121 architecture
model = models.densenet121(weights=None)
model.classifier = nn.Linear(
    model.classifier.in_features,
    NUM_CLASSES
)

# 3. Load the common Round 0 parameters
global_state = torch.load(
    GLOBAL_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(global_state, strict=True)

# 4. Keep the entire model trainable for federated fine-tuning.
# BatchNorm running statistics will be held fixed for consistency.
for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.eval()

model = model.to(DEVICE)

print("=" * 65)
print(f"NEUROLENS — MODEL INITIALIZED: {HOSPITAL_ID}")
print("=" * 65)
print("Checkpoint:", GLOBAL_CHECKPOINT.name)
print("SHA-256:", checkpoint_hash)
print("Device:", DEVICE)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Total parameters:", sum(p.numel() for p in model.parameters()))

assert model.classifier.weight.shape == (NUM_CLASSES, 1024)
assert model.features.conv0.weight.shape == (64, 3, 7, 7)

print("\nModel architecture verified.")
print("All parameters loaded with strict=True.")
print("Ready for the training-configuration cell.")

NEUROLENS — MODEL INITIALIZED: hospital_2
Checkpoint: global_model_round_000.pth
SHA-256: 5c35dbf9eafa1ef1c923b603d2d44318aadd3abf2756275b0fd159ff0c9cc754
Device: cuda
Trainable parameters: 6957956
Total parameters: 6957956

Model architecture verified.
All parameters loaded with strict=True.
Ready for the training-configuration cell.


In [10]:
# CELL 9: Training Configuration
# ALL THREE MEMBERS RUN THIS CELL

import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import json
import hashlib
import random
import numpy as np

# Set your hospital ID
HOSPITAL_ID = "hospital_2"  # hospital_1 / hospital_2 / hospital_3

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Shared Drive output structure
PROJECT_DIR = Path("/content/drive/MyDrive/Mega Project/Mega Project")
GLOBAL_DIR = PROJECT_DIR / "global_models"
OUTPUT_DIR = PROJECT_DIR / "hospital_outputs" / HOSPITAL_ID
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
REPORT_DIR = OUTPUT_DIR / "reports"

for folder in [OUTPUT_DIR, CHECKPOINT_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Training hyperparameters
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 2

# Weighted loss to address class imbalance
train_counts = np.bincount(
    np.array(train_dataset.targets),
    minlength=NUM_CLASSES
)

assert len(train_counts) == NUM_CLASSES
assert np.all(train_counts > 0), (
    "A training class has zero samples."
)

class_weights = train_counts.sum() / (
    NUM_CLASSES * train_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# AdamW optimizer for local fine-tuning
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Reduce learning rate when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

# Record training configuration
config = {
    "project": "NeuroLens",
    "hospital_id": HOSPITAL_ID,
    "model_name": "DenseNet121",
    "base_model": "global_model_round_000.pth",
    "base_model_sha256": hashlib.sha256(
        (GLOBAL_DIR / "global_model_round_000.pth").read_bytes()
    ).hexdigest(),
    "num_classes": NUM_CLASSES,
    "class_to_idx": train_dataset.class_to_idx,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "train_counts": train_counts.tolist(),
    "class_weights": class_weights.detach().cpu().tolist(),
    "seed": SEED
}

with open(OUTPUT_DIR / "training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("=" * 65)
print(f"NEUROLENS — TRAINING CONFIG: {HOSPITAL_ID}")
print("=" * 65)
print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Class counts:", train_counts.tolist())
print("Class weights:", class_weights.detach().cpu().tolist())
print("Output directory:", OUTPUT_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Report directory:", REPORT_DIR)
print("\nConfiguration saved.")
print("Training has NOT started yet.")

NEUROLENS — TRAINING CONFIG: hospital_2
Epochs: 5
Learning rate: 0.0001
Batch size: 32
Class counts: [1220, 260, 18605, 3538]
Class weights: [4.840778827667236, 22.71442222595215, 0.3174281120300293, 1.669234037399292]
Output directory: /content/drive/MyDrive/Mega Project/Mega Project/hospital_outputs/hospital_2
Checkpoint directory: /content/drive/MyDrive/Mega Project/Mega Project/hospital_outputs/hospital_2/checkpoints
Report directory: /content/drive/MyDrive/Mega Project/Mega Project/hospital_outputs/hospital_2/reports

Configuration saved.
Training has NOT started yet.


In [11]:
# CELL 10: Local DenseNet121 Training
# ALL THREE MEMBERS RUN THIS CELL

import copy
import csv
import time
import torch
import torch.nn as nn
from pathlib import Path

# Confirm the current hospital's output directory.
assert HOSPITAL_ID in {"hospital_1", "hospital_2", "hospital_3"}
assert len(train_loader.dataset) > 0
assert len(val_loader.dataset) > 0

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_local_model.pth"
HISTORY_PATH = REPORT_DIR / "training_history.csv"

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
history = []

print("=" * 70)
print(f"NEUROLENS — LOCAL TRAINING: {HOSPITAL_ID}")
print("=" * 70)

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    # -------------------------
    # TRAINING PHASE
    # -------------------------
    model.train()

    # Keep BatchNorm layers in evaluation mode to preserve
    # their running statistics during federated fine-tuning.
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        train_loss_sum += loss.item() * batch_size
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += batch_size

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # -------------------------
    # VALIDATION PHASE
    # -------------------------
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            val_loss_sum += loss.item() * batch_size
            val_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()
            val_total += batch_size

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    # Adjust learning rate using validation loss.
    scheduler.step(val_loss)

    epoch_time = time.time() - start_time

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "epoch_seconds": epoch_time,
    }
    history.append(row)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )

    # Save the best checkpoint based on validation loss.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                },
                "hospital_id": HOSPITAL_ID,
                "epoch": best_epoch,
                "best_val_loss": best_val_loss,
                "base_model": "global_model_round_000.pth",
                "class_to_idx": train_dataset.class_to_idx,
            },
            BEST_MODEL_PATH,
        )

        print("  -> Best checkpoint saved.")
    else:
        epochs_without_improvement += 1

    # Update CSV after every epoch so progress is retained.
    with open(HISTORY_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)

    # Early stopping
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping: validation loss did not improve.")
        break

print("\n" + "=" * 70)
print("LOCAL TRAINING COMPLETE")
print("=" * 70)
print("Hospital:", HOSPITAL_ID)
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))
print("Best checkpoint:", BEST_MODEL_PATH)
print("Training history:", HISTORY_PATH)

NEUROLENS — LOCAL TRAINING: hospital_2
Epoch 1/5 | Train Loss: 1.0687 | Train Acc: 0.6209 | Val Loss: 0.9517 | Val Acc: 0.5660 | Time: 269.0s
  -> Best checkpoint saved.
Epoch 2/5 | Train Loss: 0.1979 | Train Acc: 0.9008 | Val Loss: 2.0469 | Val Acc: 0.6552 | Time: 264.9s
Epoch 3/5 | Train Loss: 0.0581 | Train Acc: 0.9790 | Val Loss: 2.8691 | Val Acc: 0.6677 | Time: 264.0s
Early stopping: validation loss did not improve.

LOCAL TRAINING COMPLETE
Hospital: hospital_2
Best epoch: 1
Best validation loss: 0.9517
Best checkpoint: /content/drive/MyDrive/Mega Project/Mega Project/hospital_outputs/hospital_2/checkpoints/best_local_model.pth
Training history: /content/drive/MyDrive/Mega Project/Mega Project/hospital_outputs/hospital_2/reports/training_history.csv


In [12]:
# CELL 11: Final Local Test Evaluation
# ALL THREE MEMBERS RUN THIS CELL

import torch
import torch.nn as nn
import pandas as pd
import json
from pathlib import Path
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# Use the same HOSPITAL_ID and OUTPUT_DIR as Cell 9.
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_local_model.pth"

assert BEST_MODEL_PATH.exists(), (
    f"Best checkpoint not found: {BEST_MODEL_PATH}"
)

# Load the best checkpoint saved during training.
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

assert checkpoint["hospital_id"] == HOSPITAL_ID, (
    "Checkpoint hospital ID does not match this notebook."
)
assert checkpoint["class_to_idx"] == train_dataset.class_to_idx, (
    "Checkpoint class mapping differs from the dataset."
)

# Recreate the same architecture and load the best weights.
test_model = models.densenet121(weights=None)
test_model.classifier = nn.Linear(
    test_model.classifier.in_features,
    NUM_CLASSES
)
test_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)
test_model = test_model.to(DEVICE)
test_model.eval()

# Predict on the held-out test set.
y_true = []
y_pred = []
y_probs = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=True)

        logits = test_model(images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)

        y_true.extend(labels.cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())
        y_probs.extend(probabilities.cpu().tolist())

# Calculate evaluation metrics.
test_accuracy = accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)
test_weighted_f1 = f1_score(
    y_true, y_pred, average="weighted", zero_division=0
)

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES))
)

# Save evaluation artifacts to this hospital's output folder.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(REPORT_DIR / "test_confusion_matrix.csv")

with open(REPORT_DIR / "test_classification_report.json", "w") as f:
    json.dump(report, f, indent=2)

evaluation_summary = {
    "hospital_id": HOSPITAL_ID,
    "checkpoint_epoch": checkpoint["epoch"],
    "best_validation_loss": checkpoint["best_val_loss"],
    "test_samples": len(y_true),
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
    "test_weighted_f1": test_weighted_f1,
    "class_to_idx": checkpoint["class_to_idx"]
}

with open(REPORT_DIR / "test_evaluation_summary.json", "w") as f:
    json.dump(evaluation_summary, f, indent=2)

print("=" * 65)
print(f"NEUROLENS — TEST RESULTS: {HOSPITAL_ID}")
print("=" * 65)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Test samples:", len(y_true))
print(f"Test accuracy:    {test_accuracy:.4f}")
print(f"Test macro-F1:    {test_macro_f1:.4f}")
print(f"Test weighted-F1: {test_weighted_f1:.4f}")

print("\nPer-class classification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    zero_division=0
))

print("\nConfusion matrix:")
print(cm)

print("\nReports saved to:", REPORT_DIR)
print("Test evaluation complete. No training performed.")

NEUROLENS — TEST RESULTS: hospital_2
Checkpoint epoch: 1
Test samples: 2778
Test accuracy:    0.6847
Test macro-F1:    0.3581
Test weighted-F1: 0.6883

Per-class classification report:
                    precision    recall  f1-score   support

     Mild Dementia       0.00      0.00      0.00       244
 Moderate Dementia       1.00      0.06      0.11        33
      Non Demented       0.90      0.75      0.82      2074
Very mild Dementia       0.36      0.82      0.50       427

          accuracy                           0.68      2778
         macro avg       0.57      0.41      0.36      2778
      weighted avg       0.74      0.68      0.69      2778


Confusion matrix:
[[   0    0   93  151]
 [  21    2    0   10]
 [  62    0 1548  464]
 [   0    0   75  352]]

Reports saved to: /content/drive/MyDrive/Mega Project/Mega Project/hospital_outputs/hospital_2/reports
Test evaluation complete. No training performed.


In [13]:
# CELL 12: Verify Shared Drive Paths and Local Checkpoints
# ALL THREE MEMBERS RUN THIS CELL

from pathlib import Path
import json

HOSPITAL_ID = "hospital_2"  # change for each member

PROJECT_DIR = Path("/content/drive/MyDrive/Mega Project/Mega Project")
GLOBAL_DIR = PROJECT_DIR / "global_models"
OUTPUT_DIR = PROJECT_DIR / "hospital_outputs" / HOSPITAL_ID
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
REPORT_DIR = OUTPUT_DIR / "reports"

GLOBAL_CHECKPOINT = GLOBAL_DIR / "global_model_round_000.pth"
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_local_model.pth"
EVAL_PATH = REPORT_DIR / "test_evaluation_summary.json"

print("=" * 65)
print("NEUROLENS — SHARED ARTIFACT VERIFICATION")
print("=" * 65)
print("Hospital ID:", HOSPITAL_ID)
print("Project path:", PROJECT_DIR)
print("Global checkpoint exists:", GLOBAL_CHECKPOINT.exists())
print("Best local checkpoint exists:", BEST_MODEL_PATH.exists())
print("Test report exists:", EVAL_PATH.exists())

assert GLOBAL_CHECKPOINT.exists(), (
    "Shared Round 0 checkpoint not found at the expected path."
)
assert BEST_MODEL_PATH.exists(), (
    "Best local checkpoint not found. Check Cell 10 output path."
)

checkpoint = __import__("torch").load(
    BEST_MODEL_PATH, map_location="cpu", weights_only=True
)

print("Checkpoint hospital ID:", checkpoint["hospital_id"])
print("Best epoch:", checkpoint["epoch"])
print("State dictionary tensors:", len(checkpoint["model_state_dict"]))

assert checkpoint["hospital_id"] == HOSPITAL_ID

if EVAL_PATH.exists():
    with open(EVAL_PATH, "r", encoding="utf-8") as f:
        report = json.load(f)
    print("Test accuracy:", round(report["test_accuracy"], 4))
    print("Test macro-F1:", round(report["test_macro_f1"], 4))
else:
    print("Test evaluation report not found yet.")

print("\nVerification complete.")

NEUROLENS — SHARED ARTIFACT VERIFICATION
Hospital ID: hospital_2
Project path: /content/drive/MyDrive/Mega Project/Mega Project
Global checkpoint exists: True
Best local checkpoint exists: True
Test report exists: True
Checkpoint hospital ID: hospital_2
Best epoch: 1
State dictionary tensors: 727
Test accuracy: 0.6847
Test macro-F1: 0.3581

Verification complete.


In [14]:
# CELL 13: Export Local Model for Federated Aggregation
# ALL THREE MEMBERS RUN THIS CELL

from pathlib import Path
import torch
import json
import hashlib
from datetime import datetime, timezone

HOSPITAL_ID = "hospital_2"  # change to hospital_2 or hospital_3

PROJECT_DIR = Path("/content/drive/MyDrive/Mega Project/Mega Project")
GLOBAL_DIR = PROJECT_DIR / "global_models"
OUTPUT_DIR = PROJECT_DIR / "hospital_outputs" / HOSPITAL_ID
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
REPORT_DIR = OUTPUT_DIR / "reports"

GLOBAL_CHECKPOINT = GLOBAL_DIR / "global_model_round_000.pth"
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_local_model.pth"
EXPORT_PATH = OUTPUT_DIR / "client_weights.pth"
METADATA_PATH = OUTPUT_DIR / "client_metadata.json"

assert GLOBAL_CHECKPOINT.exists(), "Round 0 checkpoint missing."
assert BEST_MODEL_PATH.exists(), "Best local checkpoint missing."

# Verify the hospital checkpoint identity.
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

assert checkpoint["hospital_id"] == HOSPITAL_ID, (
    "Hospital ID mismatch. Do not export."
)

# Verify class mapping.
assert checkpoint["class_to_idx"] == {
    "Mild Dementia": 0,
    "Moderate Dementia": 1,
    "Non Demented": 2,
    "Very mild Dementia": 3
}, "Class mapping mismatch."

# Export only the model state dictionary.
state_dict = checkpoint["model_state_dict"]
torch.save(state_dict, EXPORT_PATH)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Read test metrics if available.
evaluation_path = REPORT_DIR / "test_evaluation_summary.json"
evaluation = None

if evaluation_path.exists():
    with open(evaluation_path, "r", encoding="utf-8") as f:
        evaluation = json.load(f)

# Use actual training-set count from Cell 9's saved config.
config_path = OUTPUT_DIR / "training_config.json"
assert config_path.exists(), "Training configuration missing."

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

metadata = {
    "project": "NeuroLens",
    "client_id": HOSPITAL_ID,
    "model_name": "DenseNet121",
    "round": 1,
    "base_model_version": "global_model_round_000",
    "num_train_examples": sum(config["train_counts"]),
    "num_classes": 4,
    "class_to_idx": config["class_to_idx"],
    "best_epoch": checkpoint["epoch"],
    "best_validation_loss": checkpoint["best_val_loss"],
    "checkpoint_sha256": sha256_file(EXPORT_PATH),
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "test_metrics": evaluation
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("=" * 65)
print(f"NEUROLENS — CLIENT EXPORT: {HOSPITAL_ID}")
print("=" * 65)
print("Weights:", EXPORT_PATH)
print("Size (MB):", round(EXPORT_PATH.stat().st_size / (1024**2), 2))
print("Metadata:", METADATA_PATH)
print("Training examples:", metadata["num_train_examples"])
print("Best epoch:", metadata["best_epoch"])
print("SHA-256:", metadata["checkpoint_sha256"])
print("\nExport complete.")

NEUROLENS — CLIENT EXPORT: hospital_2
Weights: /content/drive/MyDrive/Mega Project/Mega Project/hospital_outputs/hospital_2/client_weights.pth
Size (MB): 27.09
Metadata: /content/drive/MyDrive/Mega Project/Mega Project/hospital_outputs/hospital_2/client_metadata.json
Training examples: 23623
Best epoch: 1
SHA-256: ddd00b295c1bd6f565a560296db4f8edf83021bf760c61860d457a1ceafd5188

Export complete.
